# Feature Extraction Pipeline: Mathematical Formulation and Variable Definitions

## Objective
Transform microscopy images into a 51-dimensional feature vector $\mathbf{f} \in \mathbb{R}^{51}$ encoding geometric morphology and chromatic distribution. The pipeline enforces illumination normalization, statistical segmentation, convex geometric modeling, and ratio-based color invariance to produce interpretable, scale-consistent descriptors.

## 1. Color Normalization (Gray-World Assumption)
Illumination bias is removed using gray-world normalization. The spatial average of reflectance is assumed achromatic.

**Equations:**
$$I'_c = I_c \cdot \frac{\mu_{gray}}{\mu_c}$$
$$\mu_c = \frac{1}{N} \sum_{x,y} I_c(x,y)$$

**Variables:**
*   $c \in \{B, G, R\}$: Color channel index.
*   $I_c$: Input intensity matrix for channel $c$.
*   $I'_c$: Normalized intensity matrix for channel $c$.
*   $\mu_c$: Mean intensity of channel $c$.
*   $\mu_{gray}$: Mean intensity of the grayscale conversion of the input image.
*   $N$: Total number of pixels.
*   $x, y$: Spatial coordinates.

**Operation:** Rescales each channel to equalize global intensity distribution.

## 2. Nucleus-Enhancing Channel Construction
Multiple color spaces construct a nonlinear fusion emphasizing chromatin density.

**Equations:**
$$min\_MS = \min(M, S)$$
$$KM = K - \min(K, M)$$
$$min\_MS\_KM = min\_MS - \min(min\_MS, KM)$$

**Variables:**
*   $M$: Magenta channel from CMYK color space.
*   $K$: Black channel from CMYK color space.
*   $S$: Saturation channel from HLS color space.
*   $min\_MS$: Intermediate variable (minimum of Magenta and Saturation).
*   $KM$: Difference between Black and minimum of Black and Magenta.
*   $min\_MS\_KM$: Final enhanced channel intensity.

**Operation:** Suppresses cytoplasm and background while amplifying nucleus intensity contrast.

## 3. Gaussian Smoothing
Noise reduction is performed via Gaussian convolution.

**Equations:**
$$I_{blur}(x,y) = (I * G_\sigma)(x,y)$$
$$G_\sigma(x,y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$$

**Variables:**
*   $I$: Input intensity matrix (from Step 2).
*   $I_{blur}$: Smoothed intensity matrix.
*   $*$: Convolution operator.
*   $G_\sigma$: Gaussian kernel.
*   $\sigma$: Standard deviation controlling blur extent.
*   $x, y$: Spatial coordinates.

**Operation:** Reduces high-frequency variance prior to thresholding.

## 4. Multi-Class Otsu Segmentation
Multi-Otsu thresholding partitions intensity into three classes by minimizing intra-class variance.

**Equations:**
$$\sigma_w^2 = \sum_{k=1}^{3} \omega_k \sigma_k^2$$
$$N(x,y) = \begin{cases} 1 & I(x,y) \ge t_2 \\ 0 & \text{otherwise} \end{cases}$$

**Variables:**
*   $\sigma_w^2$: Weighted intra-class variance.
*   $k$: Class index ($1, 2, 3$).
*   $\omega_k$: Probability weight of class $k$.
*   $\sigma_k^2$: Variance of class $k$.
*   $t_1, t_2$: Computed thresholds.
*   $N(x,y)$: Binary nucleus mask.
*   $I(x,y)$: Input intensity at $(x,y)$.

**Operation:** Isolates nucleus from cytoplasm and background by retaining the highest intensity class.

## 5. Largest Connected Component
Selection of the primary structure among detected contours.

**Equation:**
$$C^* = \arg\max_{C_i} Area(C_i)$$

**Variables:**
*   $C_i$: Individual detected contour.
*   $C^*$: Selected contour with maximum area.
*   $Area(C_i)$: Spatial area enclosed by contour $C_i$.

**Operation:** Eliminates debris and neighboring structures.

## 6. Morphological Closing and Hole Filling
Topological regularization of the binary mask.

**Equation:**
$$A \bullet B = (A \oplus B) \ominus B$$

**Variables:**
*   $A$: Input binary mask.
*   $B$: Structuring element (ellipse).
*   $\bullet$: Morphological closing operator.
*   $\oplus$: Dilation operator.
*   $\ominus$: Erosion operator.

**Operation:** Fills narrow gaps. Flood-fill ensures removal of interior voids, producing a topologically solid mask.

## 7. Convex Hull Construction
Definition of convex envelope and concavity regions.

**Equations:**
$$S = \{(x,y) : N(x,y)=1\}$$
$$H = Conv(S)$$
$$NCL = S$$
$$CVX = H$$
$$ROC = H \setminus S$$

**Variables:**
*   $S$: Set of nucleus pixels.
*   $H$: Convex hull of set $S$.
*   $Conv(\cdot)$: Convex hull operation.
*   $NCL$: Nucleus region.
*   $CVX$: Convex hull region.
*   $ROC$: Region of Concavity (Hull minus Nucleus).
*   $\setminus$: Set difference operator.

**Operation:** Removes concavities while preserving the outer envelope.

## 8. Shape Descriptors
Quantification of geometric properties.

**Equations:**
$$C = \frac{P_{smooth}^2}{4\pi A}$$
$$Conv = \frac{P_{hull}}{P_{nucleus}}$$
$$Sol = \frac{A_{nucleus}}{A_{hull}}$$

**Variables:**
*   $C$: Circularity metric ($C=1$ for perfect circle).
*   $P_{smooth}$: Gaussian-smoothed nucleus perimeter.
*   $A$: Nucleus area.
*   $Conv$: Convexity metric ($Conv \in (0,1]$).
*   $P_{hull}$: Perimeter of convex hull.
*   $P_{nucleus}$: Perimeter of nucleus.
*   $Sol$: Solidity metric.
*   $A_{nucleus}$: Area of nucleus.
*   $A_{hull}$: Area of convex hull.

**Operation:** Measures area deficiency due to concavities and perimeter ratios.

## 9. Color Channels
Extraction of luminance and chromatic information from four color spaces.

**Components:**
*   RGB: $R$ (Red), $G$ (Green), $B$ (Blue).
*   HSV: $H$ (Hue), $S$ (Saturation), $V$ (Value).
*   LAB: $L$ (Lightness), $A$ (Green–Red axis), $B^*$ (Blue–Yellow axis).
*   YCrCb: $Y$ (Luminance), $Cr$ (Red difference), $Cb$ (Blue difference).

**Operation:** Provides complementary luminance and chromatic information (12 channels total).

## 10. Statistical Moments per Region
Calculation of first and second-order statistics for each channel and region.

**Equations:**
$$\mu_{R,c} = \frac{1}{|R|} \sum_{(x,y)\in R} I_c(x,y)$$
$$\sigma_{R,c} = \sqrt{\frac{1}{|R|} \sum (I_c - \mu)^2}$$

**Variables:**
*   $R \in \{NCL, ROC, CVX\}$: Region of interest.
*   $c$: Color channel index.
*   $|R|$: Cardinality (pixel count) of region $R$.
*   $I_c(x,y)$: Intensity at coordinate $(x,y)$ for channel $c$.
*   $\mu_{R,c}$: Mean intensity of region $R$ in channel $c$.
*   $\sigma_{R,c}$: Standard deviation of region $R$ in channel $c$.

**Operation:** Encodes distributional properties of color within geometric regions.

## 11. Ratio Normalization
Replacement of absolute intensities with relative measures.

**Equations:**
$$R^{mean}_{NCL,c} = \frac{\mu_{NCL,c}}{\mu_{CVX,c}}$$
$$R^{std}_{ROC,c} = \frac{\sigma_{ROC,c}}{\sigma_{CVX,c}}$$

**Variables:**
*   $R^{mean}, R^{std}$: Normalized mean and standard deviation ratios.
*   $\epsilon$: Small constant added to denominator to prevent division by zero.
*   $\mu_{CVX,c}, \sigma_{CVX,c}$: Reference statistics from the Convex Hull region.

**Operation:** Enforces illumination invariance and local contrast encoding.

## 12. Final Representation
Concatenation of all descriptors into a single vector.

**Vector:**
$$\mathbf{f} \in \mathbb{R}^{51}$$

**Components:**
*   12 NCL mean ratios.
*   12 NCL std ratios.
*   12 ROC mean ratios.
*   12 ROC std ratios.
*   3 geometric descriptors (Circularity, Convexity, Solidity).

**Operation:** Produces interpretable, scale-consistent descriptors suitable for quantitative analysis.